# Phase 0 — data prep (Colab, CPU runtime is enough)
Downloads UTKFace + FairFace-val at **pinned revisions**, verifies SHA256, stores archives on Drive,
unpacks to `/content/data`, cross-checks UTKFace labels against an independent mirror, runs
`scripts/run_phase0_counts.py`, and builds the ≤500-image dev subset.

**Ethics (EXPERIMENT_PLAN §7):** do not display any images in this notebook. Images and per-image labels stay on Drive / `/content`.

Code comes from `github.com/isaacmutuma/child-skintone-detection` (public for now). Mount Drive with the account that should hold the data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/child-detection-audit')
RAW = DRIVE_ROOT / 'raw'; RAW.mkdir(parents=True, exist_ok=True)
REPO = pathlib.Path('/content/child-detection-skin-tone-audit')
DATA = pathlib.Path('/content/data'); DATA.mkdir(exist_ok=True)
os.environ['CHILD_AUDIT_ENV'] = 'colab'

## 1. Code
Clone the repo (records the exact commit in every results JSON) and install extras.

In [ ]:
!rm -rf {REPO} && git clone -q https://github.com/isaacmutuma/child-skintone-detection.git {REPO}
!pip -q install open_clip_torch huggingface_hub
import sys; sys.path.insert(0, str(REPO)); os.chdir(REPO)
from configs import config as C
!git -C {REPO} log --oneline -1
print(C.ENV, C.DATA_DIR)

## 2. Download pinned archives to Drive and verify SHA256
Skips the download if a verified copy is already on Drive.

In [ ]:
import hashlib, json, shutil
from huggingface_hub import hf_hub_download

def sha256(p, bs=1<<20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(bs), b''): h.update(b)
    return h.hexdigest()

prov = {}
for name, s in C.SOURCES.items():
    dst = RAW / pathlib.Path(s['file']).name
    if not (dst.exists() and sha256(dst) == s['sha256']):
        p = hf_hub_download(s['hf_repo'], s['file'], repo_type='dataset', revision=s['revision'])
        shutil.copy(p, dst)
    got = sha256(dst)
    assert got == s['sha256'], f'{name}: SHA256 mismatch {got}'
    prov[name] = {**s, 'local_file': str(dst), 'bytes': dst.stat().st_size, 'sha256_verified': True}
    print(name, 'OK', dst.stat().st_size)

## 3. Unpack to `/content/data`

In [ ]:
import tarfile, io, csv
import pyarrow.parquet as pq

# UTKFace: safe extraction (regular files only, no paths escaping the target)
utk_tar = RAW / 'UTKFace.tar.gz'
out = DATA
with tarfile.open(utk_tar) as t:
    members = [m for m in t.getmembers() if m.isfile()
               and not m.name.startswith('/') and '..' not in pathlib.PurePosixPath(m.name).parts]
    t.extractall(out, members=members)
cands = [p for p in out.iterdir() if p.is_dir() and p.name.lower().startswith('utkface')]
print('extracted dirs:', [c.name for c in cands])
if cands and cands[0].name != 'UTKFace': cands[0].rename(out / 'UTKFace')
n_utk = sum(1 for _ in (out/'UTKFace').iterdir())
print('UTKFace files:', n_utk)

# FairFace val: parquet -> val/<n>.jpg + fairface_label_val.csv (official layout)
AGE = ['0-2','3-9','10-19','20-29','30-39','40-49','50-59','60-69','more than 70']
GENDER = ['Male','Female']
RACE = ['East Asian','Indian','Black','White','Middle Eastern','Latino_Hispanic','Southeast Asian']
ff = DATA/'fairface'; (ff/'val').mkdir(parents=True, exist_ok=True)
tbl = pq.read_table(RAW / pathlib.Path(C.SOURCES['fairface_val']['file']).name).to_pylist()
with open(ff/'fairface_label_val.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['file','age','gender','race','service_test'])
    for i, r in enumerate(tbl, start=1):
        # HF rows carry no original file name; rows are numbered in parquet order (val/1.jpg ...)
        rel = f'val/{i}.jpg'
        (ff/rel).write_bytes(r['image']['bytes'])
        w.writerow([rel, AGE[r['age']], GENDER[r['gender']], RACE[r['race']], r['service_test']])
print('FairFace val rows:', len(tbl))
prov['utkface']['n_files_extracted'] = n_utk
prov['fairface_val']['n_rows'] = len(tbl)

## 4. Cross-check UTKFace labels against an independent mirror
Reads only the label columns of `nu-delta/utkface` (no images). Compares the file-name sets and
the age/gender/race parsed from names. If this cell fails (mirror gone), note it in the data card and continue.

In [ ]:
from huggingface_hub import HfFileSystem
from src.data import load_utkface
try:
    fs = HfFileSystem()
    files = sorted(fs.glob(f"datasets/{C.SOURCES['utkface']['crosscheck_repo']}/data/*.parquet"))
    names = set()
    for fpath in files:
        with fs.open(fpath) as fh:
            names |= set(pq.read_table(fh, columns=['file_name']).column('file_name').to_pylist())
    ours = {p.name for p in (DATA/'UTKFace').iterdir()}
    strip = lambda n: n.split('.')[0]
    a, b = {strip(n) for n in ours}, {strip(pathlib.Path(n).name) for n in names}
    cc = {'mirror': C.SOURCES['utkface']['crosscheck_repo'], 'n_ours': len(a), 'n_mirror': len(b),
          'n_common': len(a & b), 'only_ours': len(a - b), 'only_mirror': len(b - a)}
except Exception as e:
    cc = {'error': repr(e)}
print(cc); prov['utkface']['crosscheck'] = cc
recs, skipped = load_utkface()
print('parsed records:', len(recs), 'unparseable names:', len(skipped))
prov['utkface']['n_unparseable_names'] = len(skipped)

## 5. Counts + power check → `results/phase0_counts.json`; provenance → `results/data_provenance.json`

In [ ]:
!python scripts/run_phase0_counts.py
!python scripts/write_env.py
from src.provenance import write_result
write_result(REPO/'results'/'data_provenance.json', {'sources': prov}, {'ENV': C.ENV}, REPO)
res_drive = DRIVE_ROOT/'results'; res_drive.mkdir(exist_ok=True)
for n in ['phase0_counts.json', 'data_provenance.json']:
    shutil.copy(REPO/'results'/n, res_drive/n)
print('copied to', res_drive)

## 6. Dev subset (≤500 images) for the Mac
Download `dev_subset.zip` from Drive and unzip into `data/dev/` in the local repo.

In [ ]:
!python scripts/make_dev_subset.py --out {DRIVE_ROOT}/dev_subset.zip
!ls -la {DRIVE_ROOT}